In [1]:
import pandas as pd

In [5]:
get_decoder = lambda x: [i["config"].split("_")[0] for i in x.to_dict(orient="records")]

In [18]:
if_cfenet = lambda x: ["cfenet" in i["config"] for i in x.to_dict(orient="records")]

In [47]:
dataset_map = {
    "mas": "Massachusetts",
    "whu": "WHU",
}
    
get_training_dataset = lambda x: [dataset_map.get(i["config"].split("_")[1]) for i in x.to_dict(orient="records")]

#Seed 42 Benchmarks

In [26]:
def assign_name(name):
    map_names = {
        "cfenet": "CFENet",
        "unet": "UNet",
        "upernet": "UPerNet",
        "dlab": "DeepLabV3+"
    }

    name_parts = name.split("_")
    if_cfenet = "cfenet" in name_parts
    decoder = name_parts[0]
    return " + ".join([map_names[decoder], "CFENet"]) if if_cfenet else map_names[decoder]

In [48]:
benchmarks = pd.read_csv("seed42_benchmarks.csv")
benchmarks["decoder"] = get_decoder(benchmarks)
benchmarks["cfenet"] = if_cfenet(benchmarks)
benchmarks["training_dataset"] = get_training_dataset(benchmarks)
benchmarks["name"] = [assign_name(name) for name in benchmarks["config"].to_list()]
benchmarks

,config,dataset,timestamp,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou,tp,fp,fn,tn,decoder,cfenet,training_dataset,name
0,unet_mas_final_best,WHU Test,26-06-2026 13:14:41,0.709321,0.837496,0.822529,0.829945,0.947685,0.940029,0.824675,41095783,7974055,8866921,263976073,unet,False,Massachusetts,UNet
1,unet_mas_final_best,Massachusetts,26-06-2026 13:14:41,0.736752,0.843311,0.853602,0.848425,0.943250,0.932535,0.834643,3573594,663981,612894,17649531,unet,False,Massachusetts,UNet
2,unet_mas_cfenet_final_best,WHU Test,26-06-2026 13:20:57,0.711671,0.801365,0.864100,0.831551,0.945665,0.937249,0.824460,43172789,10701286,6789915,261248842,unet,True,Massachusetts,UNet + CFENet
3,unet_mas_cfenet_final_best,Massachusetts,26-06-2026 13:20:57,0.750145,0.854323,0.860172,0.857237,0.946692,0.936534,0.843339,3601100,614051,585388,17699461,unet,True,Massachusetts,UNet + CFENet
4,unet_whu_final_best,WHU Test,26-06-2026 13:23:56,0.911117,0.956384,0.950616,0.953491,0.985607,0.983116,0.947116,47495339,2166007,2467365,269784121,unet,False,WHU,UNet
5,unet_whu_final_best,Massachusetts,26-06-2026 13:23:56,0.562983,0.804875,0.651966,0.720395,0.905834,0.892834,0.727908,2729447,661697,1457041,17651815,unet,False,WHU,UNet
6,unet_whu_cfenet_tmax100_best,WHU Test,25-06-2026 12:43:52,0.919213,0.960005,0.955816,0.957906,0.986962,0.984691,0.951952,47755158,1989526,2207546,269960602,unet,True,WHU,UNet + CFENet
7,unet_whu_cfenet_tmax100_best,Massachusetts,25-06-2026 12:43:52,0.541345,0.824296,0.611959,0.702431,0.903528,0.891131,0.716238,2561961,546100,1624527,17767412,unet,True,WHU,UNet + CFENet
8,dlab_mas_main_best,WHU Test,03-07-2026 22:28:28,0.752673,0.839132,0.879592,0.858886,0.955141,0.948048,0.850361,43946804,8424919,6015900,263525209,dlab,False,Massachusetts,DeepLabV3+
9,dlab_mas_main_best,Massachusetts,03-07-2026 22:28:28,0.754789,0.867374,0.853265,0.860261,0.948422,0.938688,0.846738,3572185,546208,614303,17767304,dlab,False,Massachusetts,DeepLabV3+


In [56]:
upernet_final_configs = ["upernet_mas_main_best", "upernet_whu_main_best", "upernet_mas_cfenet_final-2_best", "upernet_whu_cfenet_final_best"]

In [63]:
# I want all configs except those where decoder=upernet and they are not in the upernet_final_configs list.
presentable_benchmarks = benchmarks[(benchmarks["decoder"] != "upernet") | benchmarks["config"].isin(upernet_final_configs)]
presentable_benchmarks.head()

,config,dataset,timestamp,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou,tp,fp,fn,tn,decoder,cfenet,training_dataset,name
0,unet_mas_final_best,WHU Test,26-06-2026 13:14:41,0.709321,0.837496,0.822529,0.829945,0.947685,0.940029,0.824675,41095783,7974055,8866921,263976073,unet,False,Massachusetts,UNet
1,unet_mas_final_best,Massachusetts,26-06-2026 13:14:41,0.736752,0.843311,0.853602,0.848425,0.943250,0.932535,0.834643,3573594,663981,612894,17649531,unet,False,Massachusetts,UNet
2,unet_mas_cfenet_final_best,WHU Test,26-06-2026 13:20:57,0.711671,0.801365,0.864100,0.831551,0.945665,0.937249,0.824460,43172789,10701286,6789915,261248842,unet,True,Massachusetts,UNet + CFENet
3,unet_mas_cfenet_final_best,Massachusetts,26-06-2026 13:20:57,0.750145,0.854323,0.860172,0.857237,0.946692,0.936534,0.843339,3601100,614051,585388,17699461,unet,True,Massachusetts,UNet + CFENet
4,unet_whu_final_best,WHU Test,26-06-2026 13:23:56,0.911117,0.956384,0.950616,0.953491,0.985607,0.983116,0.947116,47495339,2166007,2467365,269784121,unet,False,WHU,UNet


In [58]:
clean_table = presentable_benchmarks.iloc[:, [-1, 16, 1, 3, 4, 5, 6, 7, 8, 9,]]

In [59]:
whu_benchmarks = clean_table[clean_table["dataset"] == "WHU Test"]
whu_benchmarks = whu_benchmarks.drop(columns=["dataset"])
# Give a heading that these are the results on the WHU test set
print("Results on the WHU test set:")
whu_benchmarks

Results on the WHU test set:


,name,training_dataset,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou
0,UNet,Massachusetts,0.709321,0.837496,0.822529,0.829945,0.947685,0.940029,0.824675
2,UNet + CFENet,Massachusetts,0.711671,0.801365,0.864100,0.831551,0.945665,0.937249,0.824460
4,UNet,WHU,0.911117,0.956384,0.950616,0.953491,0.985607,0.983116,0.947116
6,UNet + CFENet,WHU,0.919213,0.960005,0.955816,0.957906,0.986962,0.984691,0.951952
8,DeepLabV3+,Massachusetts,0.752673,0.839132,0.879592,0.858886,0.955141,0.948048,0.850361
10,DeepLabV3+ + CFENet,Massachusetts,0.728780,0.798115,0.893492,0.843115,0.948391,0.940082,0.834431
12,DeepLabV3+,WHU,0.897726,0.950639,0.941618,0.946106,0.983350,0.980501,0.939113
14,DeepLabV3+ + CFENet,WHU,0.910144,0.959407,0.946597,0.952958,0.985495,0.982997,0.946571
16,UPerNet,WHU,0.915973,0.959620,0.952693,0.956143,0.986436,0.984082,0.950027
18,UPerNet,Massachusetts,0.727727,0.851338,0.833666,0.842409,0.951590,0.944395,0.836061


In [60]:
mas_benchmarks = clean_table[clean_table["dataset"] == "Massachusetts"]
mas_benchmarks = mas_benchmarks.drop(columns=["dataset"])
print("Results on the Massachusetts test set:")
mas_benchmarks

Results on the Massachusetts test set:


,name,training_dataset,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou
1,UNet,Massachusetts,0.736752,0.843311,0.853602,0.848425,0.943250,0.932535,0.834643
3,UNet + CFENet,Massachusetts,0.750145,0.854323,0.860172,0.857237,0.946692,0.936534,0.843339
5,UNet,WHU,0.562983,0.804875,0.651966,0.720395,0.905834,0.892834,0.727908
7,UNet + CFENet,WHU,0.541345,0.824296,0.611959,0.702431,0.903528,0.891131,0.716238
9,DeepLabV3+,Massachusetts,0.754789,0.867374,0.853265,0.860261,0.948422,0.938688,0.846738
11,DeepLabV3+ + CFENet,Massachusetts,0.764210,0.867770,0.864931,0.866348,0.950345,0.940821,0.852516
13,DeepLabV3+,WHU,0.432329,0.846025,0.469252,0.603673,0.885355,0.874388,0.653359
15,DeepLabV3+ + CFENet,WHU,0.463651,0.838948,0.508951,0.633554,0.890453,0.878994,0.671323
17,UPerNet,WHU,0.576394,0.790228,0.680519,0.731281,0.906943,0.893451,0.734923
19,UPerNet,Massachusetts,0.757259,0.866063,0.857705,0.861863,0.948843,0.939129,0.848194


In [62]:
inria_benchmarks = pd.read_csv("seed42_inria_benchmarks.csv")
inria_benchmarks["decoder"] = get_decoder(inria_benchmarks)
inria_benchmarks["cfenet"] = if_cfenet(inria_benchmarks)
inria_benchmarks["training_dataset"] = get_training_dataset(inria_benchmarks)
inria_benchmarks["name"] = [assign_name(name) for name in inria_benchmarks["config"].to_list()]
inria_benchmarks

,config,dataset,timestamp,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou,tp,fp,fn,tn,decoder,cfenet,training_dataset,name
0,unet_mas_final_best,austin,26-06-2026 13:30:39,0.529497,0.767098,0.630927,0.692380,0.925209,0.918335,0.723916,10521117,3194368,6154527,105129988,unet,False,Massachusetts,UNet
1,unet_mas_final_best,chicago,26-06-2026 13:30:39,0.569745,0.663639,0.801072,0.725908,0.877439,0.853694,0.711720,20286965,10282304,5037798,89392933,unet,False,Massachusetts,UNet
2,unet_mas_final_best,kitsap,26-06-2026 13:30:39,0.432004,0.623227,0.584713,0.603355,0.984576,0.984392,0.708198,1466431,886535,1041519,121605515,unet,False,Massachusetts,UNet
3,unet_mas_final_best,tyrolw,26-06-2026 13:30:39,0.612451,0.820435,0.707255,0.759652,0.967565,0.965813,0.789132,6407158,1402305,2652032,114538505,unet,False,Massachusetts,UNet
4,unet_mas_final_best,vienna,26-06-2026 13:30:39,0.611986,0.799383,0.723034,0.759294,0.880323,0.852478,0.732232,23594604,5921417,9038159,86445820,unet,False,Massachusetts,UNet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,upernet_mas_cfenet_final-2_best,chicago,04-07-2026 09:04:33,0.567027,0.714902,0.732713,0.723698,0.886649,0.866889,0.716958,18555782,7399907,6768981,92275330,upernet,True,Massachusetts,UPerNet + CFENet
86,upernet_mas_cfenet_final-2_best,kitsap,04-07-2026 09:04:33,0.332186,0.695706,0.388655,0.498707,0.984324,0.984200,0.658193,974727,426334,1533223,122065716,upernet,True,Massachusetts,UPerNet + CFENet
87,upernet_mas_cfenet_final-2_best,tyrolw,04-07-2026 09:04:33,0.641703,0.806421,0.758549,0.781752,0.969305,0.967519,0.804611,6871841,1649563,2187349,114291247,upernet,True,Massachusetts,UPerNet + CFENet
88,upernet_mas_cfenet_final-2_best,vienna,04-07-2026 09:04:33,0.626335,0.814788,0.730313,0.770240,0.886256,0.859461,0.742898,23832115,5417356,8800648,86949881,upernet,True,Massachusetts,UPerNet + CFENet


In [68]:
pre_in_benchmarks = inria_benchmarks[(inria_benchmarks["decoder"] != "upernet") | inria_benchmarks["config"].isin(upernet_final_configs)]
pre_in_benchmarks.config.unique()

array(['unet_mas_final_best', 'unet_mas_cfenet_final_best',
       'unet_whu_final_best', 'unet_whu_cfenet_tmax100_best',
       'dlab_mas_main_best', 'dlab_mas_cfenet_final-2_best',
       'dlab_whu_main_best', 'dlab_whu_cfenet_main_best',
       'upernet_whu_main_best', 'upernet_mas_main_best',
       'upernet_whu_cfenet_final_best', 'upernet_mas_cfenet_final-2_best'],
      dtype=object)

In [69]:
clean_inria_benchmarks = pre_in_benchmarks.iloc[:, [-1, 16, 1, 3, 4, 5, 6, 7, 8, 9,]]
clean_inria_benchmarks

,name,training_dataset,dataset,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou
0,UNet,Massachusetts,austin,0.529497,0.767098,0.630927,0.692380,0.925209,0.918335,0.723916
1,UNet,Massachusetts,chicago,0.569745,0.663639,0.801072,0.725908,0.877439,0.853694,0.711720
2,UNet,Massachusetts,kitsap,0.432004,0.623227,0.584713,0.603355,0.984576,0.984392,0.708198
3,UNet,Massachusetts,tyrolw,0.612451,0.820435,0.707255,0.759652,0.967565,0.965813,0.789132
4,UNet,Massachusetts,vienna,0.611986,0.799383,0.723034,0.759294,0.880323,0.852478,0.732232
...,...,...,...,...,...,...,...,...,...,...
85,UPerNet + CFENet,Massachusetts,chicago,0.567027,0.714902,0.732713,0.723698,0.886649,0.866889,0.716958
86,UPerNet + CFENet,Massachusetts,kitsap,0.332186,0.695706,0.388655,0.498707,0.984324,0.984200,0.658193
87,UPerNet + CFENet,Massachusetts,tyrolw,0.641703,0.806421,0.758549,0.781752,0.969305,0.967519,0.804611
88,UPerNet + CFENet,Massachusetts,vienna,0.626335,0.814788,0.730313,0.770240,0.886256,0.859461,0.742898


In [72]:
print("Results on the overall Inria benchmark:")
clean_inria_benchmarks[clean_inria_benchmarks["dataset"] == "overall"].drop(columns=["dataset"])

Results on the overall Inria benchmark:


,name,training_dataset,pos_iou,precision,recall,f1,accuracy,neg_iou,mean_iou
5,UNet,Massachusetts,0.577235,0.741709,0.722460,0.731958,0.927022,0.918946,0.748090
11,UNet + CFENet,Massachusetts,0.568244,0.686032,0.767962,0.724688,0.919523,0.909989,0.739117
17,UNet,WHU,0.626278,0.747367,0.794469,0.770198,0.934614,0.926567,0.776423
23,UNet + CFENet,WHU,0.653683,0.819996,0.763198,0.790578,0.944234,0.937673,0.795678
29,DeepLabV3+,Massachusetts,0.583036,0.749480,0.724164,0.736604,0.928572,0.920646,0.751841
35,DeepLabV3+ + CFENet,Massachusetts,0.601038,0.738387,0.763659,0.750810,0.930087,0.921857,0.761447
41,DeepLabV3+,WHU,0.554419,0.718053,0.708699,0.713345,0.921444,0.912934,0.733676
47,DeepLabV3+ + CFENet,WHU,0.570890,0.806040,0.661805,0.726836,0.931392,0.924500,0.747695
53,UPerNet,WHU,0.660103,0.796151,0.794361,0.795255,0.943586,0.936645,0.798374
59,UPerNet,Massachusetts,0.559306,0.762863,0.677011,0.717377,0.926428,0.918851,0.739078


In [73]:
df_cities = (
    clean_inria_benchmarks[clean_inria_benchmarks["dataset"] != "overall"]
    .pivot(index="name", columns="dataset", values="pos_iou")
    .reset_index()
)

print(df_cities)

ValueError: Index contains duplicate entries, cannot reshape

In [76]:
import pandas as pd

def pivot_city_metrics(df, metric_col):
    """
    Reshape model performance metrics by city.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Must contain columns: 'name', 'training_dataset', 'dataset', and the metric column.
    metric_col : str
        Name of the metric column to pivot (e.g., 'pos_iou', 'precision', 'recall', etc.)
    
    Returns
    -------
    pandas.DataFrame
        Columns: 'name', 'training_dataset', then each city (austin, chicago, kitsap, tyrolw, vienna)
        as separate columns with the corresponding metric values.
        Rows correspond to each unique (name, training_dataset) pair.
        The 'overall' dataset row is excluded.
    """
    # Define the city names (exclude 'overall')
    cities = ['austin', 'chicago', 'kitsap', 'tyrolw', 'vienna']
    
    # Filter to only city rows
    df_cities = df[df['dataset'].isin(cities)].copy()
    
    # Pivot: index = (name, training_dataset), columns = dataset, values = metric_col
    pivoted = df_cities.pivot_table(
        index=['name', 'training_dataset'],
        columns='dataset',
        values=metric_col,
        aggfunc='first'  # in case of duplicates, keeps first; use 'mean' if appropriate
    ).reset_index()
    
    # Ensure city columns are in the desired order
    pivoted = pivoted[['name', 'training_dataset'] + cities]
    
    return pivoted

In [78]:
print("INRIA City-wise Positive IoU):")
pivot_city_metrics(clean_inria_benchmarks, 'pos_iou')

INRIA City-wise Positive IoU):


dataset,name,training_dataset,austin,chicago,kitsap,tyrolw,vienna
0,DeepLabV3+,Massachusetts,0.552387,0.573338,0.386812,0.632358,0.614818
1,DeepLabV3+,WHU,0.550483,0.599824,0.438734,0.393173,0.605348
2,DeepLabV3+ + CFENet,Massachusetts,0.564975,0.575911,0.404211,0.628934,0.648041
3,DeepLabV3+ + CFENet,WHU,0.607989,0.550492,0.538189,0.548811,0.579677
4,UNet,Massachusetts,0.529497,0.569745,0.432004,0.612451,0.611986
5,UNet,WHU,0.614169,0.623708,0.494261,0.508392,0.693057
6,UNet + CFENet,Massachusetts,0.493858,0.567029,0.265221,0.596528,0.642736
7,UNet + CFENet,WHU,0.660732,0.623029,0.621663,0.669588,0.674247
8,UPerNet,Massachusetts,0.534042,0.565493,0.386584,0.615523,0.564749
9,UPerNet,WHU,0.669876,0.624853,0.626791,0.652238,0.690946


In [79]:
print("INRIA city-wise F1:")
pivot_city_metrics(clean_inria_benchmarks, 'f1')

INRIA city-wise F1:


dataset,name,training_dataset,austin,chicago,kitsap,tyrolw,vienna
0,DeepLabV3+,Massachusetts,0.711661,0.728817,0.557842,0.774778,0.761470
1,DeepLabV3+,WHU,0.710079,0.749862,0.609889,0.564428,0.754164
2,DeepLabV3+ + CFENet,Massachusetts,0.722024,0.730892,0.575712,0.772203,0.786437
3,DeepLabV3+ + CFENet,WHU,0.756210,0.710087,0.699769,0.708686,0.733918
4,UNet,Massachusetts,0.692380,0.725908,0.603355,0.759652,0.759294
5,UNet,WHU,0.760972,0.768251,0.661545,0.674085,0.818705
6,UNet + CFENet,Massachusetts,0.661184,0.723699,0.419249,0.747281,0.782518
7,UNet + CFENet,WHU,0.795711,0.767736,0.766698,0.802099,0.805433
8,UPerNet,Massachusetts,0.696254,0.722447,0.557606,0.762010,0.721839
9,UPerNet,WHU,0.802306,0.769119,0.770585,0.789520,0.817230
